<a href="https://colab.research.google.com/github/sanmquin/AI/blob/main/src/Graphiko/Channel-Video-Manifold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Channel Video Manifold Analysis
This notebook models each channel as a manifold (a distribution over embeddings) rather than a single point. It calculates the internal geometry of these manifolds (centers, modes, dense regions, and principal directions) and measures individual videos relative to this local geometry and an external subscription graph field.


## Environment setup and data connections

In [ ]:
# Install runtime dependencies
!pip install -q pymongo dnspython pinecone pandas numpy matplotlib seaborn scikit-learn

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)
    client.admin.command('ping')
    print('✅ Successfully connected to MongoDB!')
except Exception as e:
    print(f'❌ MongoDB connection failed: {e}')

## Fetch business cluster and channel metadata

In [ ]:
# Access Finder DB collections and identify the latest business cluster
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']
videos_col = db['videos']

latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None
print('Latest cluster version:', latest_version)

if latest_version is None:
    print('No clusters found.')
    business_cluster = None
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })

if business_cluster:
    business_cluster_id = business_cluster['_id']
    print(f"Found business cluster '{business_cluster['name']}' (version {business_cluster['version']})")
else:
    business_cluster_id = None
    print('No business cluster found.')

# Resolve channels
if business_cluster_id is None:
    print('Cannot fetch channels without a business cluster id.')
    channel_ids = []
    business_channels = []
else:
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs]

    business_channels = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1, 'description': 1}
    ))
    print(f'Found {len(business_channels)} channels in the business cluster.')

## Fetch videos for channels

In [ ]:
from collections import defaultdict
import pandas as pd

if not channel_ids:
    print('No channel ids available to query videos.')
    videos_by_channel = {}
else:
    cursor = videos_col.find(
        {'channelId': {'$in': channel_ids}},
        {
            '_id': 0,
            'videoId': 1,
            'channelId': 1,
            'title': 1,
            'publishedAt': 1,
            'viewCount': 1
        }
    ).sort([('channelId', 1), ('publishedAt', -1)])

    videos_by_channel = defaultdict(list)
    for doc in cursor:
        videos_by_channel[doc['channelId']].append(doc)

    print(f'Fetched videos for {len(videos_by_channel)} channels.')
    total_videos = sum(len(v) for v in videos_by_channel.values())
    print(f'Total videos fetched: {total_videos}')

all_videos = [v for vids in videos_by_channel.values() for v in vids] if videos_by_channel else []
videos_df = pd.DataFrame(all_videos)
videos_df['publishedAt'] = pd.to_datetime(videos_df['publishedAt'])
videos_df['viewCount'] = pd.to_numeric(videos_df['viewCount'], errors='coerce')
print('videos_df shape:', videos_df.shape)
videos_df.head(5)

## Pinecone Embedding Fetch/Embed

In [ ]:
from pinecone import Pinecone
import numpy as np

try:
    pinecone_api_key = userdata.get('PINECONE_API_KEY')
    pc = Pinecone(api_key=pinecone_api_key)
    pinecone_index = pc.Index('finder')
    print('✅ Connected to Pinecone index: finder')
except Exception as e:
    print(f'❌ Pinecone connection failed: {e}')

def fetch_or_embed_texts(pc_client, index, namespace, id_to_text, model="multilingual-e5-large", batch_size=96):
    normalized = {str(k): v for k, v in id_to_text.items() if v is not None and str(v).strip()}
    if not normalized:
        return {}
    
    ids = list(normalized.keys())
    embeddings = {}
    
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i+batch_size]
        try:
            fetch_res = index.fetch(ids=batch_ids, namespace=namespace)
            fetched = fetch_res.get('vectors', {})
            for fetched_id, vector_data in fetched.items():
                if 'values' in vector_data:
                    embeddings[fetched_id] = vector_data['values']
        except Exception as e:
            print(f'Warning: Fetch failed for namespace {namespace}: {e}')
            
    missing_ids = [vid for vid in ids if vid not in embeddings]
    if missing_ids:
        print(f"Embedding {len(missing_ids)} missing texts in namespace {namespace}...")
        for i in range(0, len(missing_ids), batch_size):
            batch_missing = missing_ids[i:i+batch_size]
            batch_texts = [normalized[vid] for vid in batch_missing]
            try:
                emb_res = pc_client.inference.embed(model=model, inputs=batch_texts, parameters={"input_type": "passage", "truncate": "END"})
                rows = getattr(emb_res, 'data', emb_res)
                
                to_upsert = []
                for rid, row in zip(batch_missing, rows):
                    values = row.get("values") if isinstance(row, dict) else getattr(row, "values", None)
                    if values is not None:
                        embeddings[rid] = values
                        to_upsert.append({'id': rid, 'values': values, 'metadata': {'text': normalized[rid][:1000]}})
                if to_upsert:
                    index.upsert(vectors=to_upsert, namespace=namespace)
            except Exception as e:
                print(f"Failed embedding batch: {e}")
                
    return embeddings

video_title_text = {
    str(v.get('videoId')): (v.get('title') or '').strip() 
    for v in all_videos if v.get('videoId') and v.get('title')
}

video_embeddings_dict = fetch_or_embed_texts(
    pc_client=pc,
    index=pinecone_index,
    namespace='VideoTitles',
    id_to_text=video_title_text
)
print(f'Video embeddings loaded: {len(video_embeddings_dict)}')


## Channel Internal Geometry (Manifold Modeling)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

def compute_recency_weights(dates, decay_rate=0.01):
    '''Calculate exponential decay weights based on recency in days.'''
    dates = pd.to_datetime(dates)
    max_date = dates.max()
    days_diff = (max_date - dates).dt.days
    weights = np.exp(-decay_rate * days_diff)
    return weights / weights.sum()

channel_manifolds = {}

for cid, group in videos_df.groupby('channelId'):
    group = group.dropna(subset=['publishedAt'])
    if len(group) < 5:
        continue
        
    vids = group['videoId'].astype(str).tolist()
    valid_vids = [v for v in vids if v in video_embeddings_dict]
    if len(valid_vids) < 5:
        continue
        
    embeddings_matrix = np.array([video_embeddings_dict[v] for v in valid_vids])
    dates = group.set_index('videoId').loc[valid_vids, 'publishedAt']
    
    # Recency-weighted centroid
    weights = compute_recency_weights(dates).values.reshape(-1, 1)
    centroid = np.sum(embeddings_matrix * weights, axis=0)
    
    # Modes (KMeans)
    n_clusters = min(3, len(valid_vids) // 5 + 1)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    kmeans.fit(embeddings_matrix)
    modes = kmeans.cluster_centers_
    
    # Local PCA (Anisotropy)
    pca = PCA(n_components=min(5, embeddings_matrix.shape[0]-1))
    # We can center by centroid, but standard PCA centers by mean. We use standard PCA for simplicity of principal axes.
    pca.fit(embeddings_matrix)
    principal_components = pca.components_
    explained_variance = pca.explained_variance_ratio_
    
    channel_manifolds[cid] = {
        'centroid': centroid,
        'modes': modes,
        'pca_components': principal_components,
        'valid_vids': valid_vids,
        'embeddings': embeddings_matrix
    }

print(f"Computed internal geometry for {len(channel_manifolds)} channels.")


## Video-Level Measurements & Channel-Relative Performance

In [ ]:
import scipy.stats as stats
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning)

video_measurements = []

for cid, manifold in channel_manifolds.items():
    valid_vids = manifold['valid_vids']
    embeddings = manifold['embeddings']
    centroid = manifold['centroid']
    modes = manifold['modes']
    pca_comp = manifold['pca_components']
    
    # Distance to recency-weighted centroid
    dist_to_centroid = np.linalg.norm(embeddings - centroid, axis=1)
    
    # Distance to nearest mode
    dist_to_modes = pairwise_distances(embeddings, modes)
    dist_to_nearest_mode = np.min(dist_to_modes, axis=1)
    
    # Local density (mean distance to k=3 nearest neighbors)
    k = min(4, len(valid_vids)) # k=4 means 3 neighbors + self
    dist_matrix = pairwise_distances(embeddings)
    np.fill_diagonal(dist_matrix, np.inf)
    dist_sorted = np.sort(dist_matrix, axis=1)
    local_density = np.mean(dist_sorted[:, :k-1], axis=1) # smaller = denser
    
    # Projection onto main PCA axis (directional component)
    if len(pca_comp) > 0:
        main_axis = pca_comp[0]
        # Project vector from centroid onto main axis
        projection_main_axis = np.dot(embeddings - centroid, main_axis)
    else:
        projection_main_axis = np.zeros(len(valid_vids))
        
    group_df = videos_df[videos_df['videoId'].isin(valid_vids)].set_index('videoId')
    
    # Calculate channel-relative performance (z-score of log views)
    views = group_df.loc[valid_vids, 'viewCount'].values
    valid_views = views > 0
    log_views = np.full_like(views, np.nan, dtype=float)
    log_views[valid_views] = np.log1p(views[valid_views])
    
    if np.sum(valid_views) > 2:
        relative_performance_z = stats.zscore(log_views, nan_policy='omit')
    else:
        relative_performance_z = np.full_like(views, np.nan, dtype=float)
        
    for i, vid in enumerate(valid_vids):
        video_measurements.append({
            'videoId': vid,
            'channelId': cid,
            'dist_to_centroid': dist_to_centroid[i],
            'dist_to_nearest_mode': dist_to_nearest_mode[i],
            'local_density_dist': local_density[i],
            'projection_main_axis': projection_main_axis[i],
            'relative_performance_z': relative_performance_z[i],
            'viewCount': views[i]
        })

measurements_df = pd.DataFrame(video_measurements)
print(f"Generated measurements for {len(measurements_df)} videos.")
measurements_df.head()


## Subscription Graph Overlay (External Affinity Field)

In [ ]:
from pathlib import Path

# Load subscription distance matrix
sub_matrix_path = Path('/content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance/latest/adjacency_matrix.csv')
if not sub_matrix_path.exists():
    print(f"Warning: Subscription matrix not found at {sub_matrix_path}")
    print("Skipping graph overlay calculation.")
else:
    sub_df = pd.read_csv(sub_matrix_path, index_col=0)
    sub_df.index = sub_df.index.map(str)
    sub_df.columns = sub_df.columns.map(str)

    # We need channel descriptions to build the affinity vectors
    channel_desc_text = {
        str(ch.get('channelId')): (ch.get('description') or '').strip()
        for ch in business_channels
        if ch.get('channelId') and (ch.get('description') or '').strip()
    }
    
    channel_embeddings_dict = fetch_or_embed_texts(
        pc_client=pc,
        index=pinecone_index,
        namespace='ChannelDescriptions',
        id_to_text=channel_desc_text
    )
    
    affinity_scores = []
    
    for _, row in measurements_df.iterrows():
        vid = row['videoId']
        cid = row['channelId']
        
        if cid not in sub_df.index or vid not in video_embeddings_dict:
            affinity_scores.append(np.nan)
            continue
            
        vid_emb = np.array(video_embeddings_dict[vid])
        centroid = channel_manifolds[cid]['centroid']
        vid_movement = vid_emb - centroid # Direction video takes away from center
        
        norm_movement = np.linalg.norm(vid_movement)
        if norm_movement == 0:
            affinity_scores.append(0)
            continue
            
        # Get subscription affinities for this channel (invert distance to get affinity)
        outbound_distances = sub_df.loc[cid]
        # Ignore self
        outbound_distances = outbound_distances.drop(labels=[cid], errors='ignore')
        # Only keep channels we have embeddings for
        valid_targets = [t for t in outbound_distances.index if t in channel_embeddings_dict]
        
        if not valid_targets:
            affinity_scores.append(np.nan)
            continue
            
        distances = outbound_distances[valid_targets].values
        # Affinity = 1 - distance (assuming normalized distance [0,1])
        affinities = 1.0 - distances
        # Normalize affinities to sum to 1
        if np.sum(affinities) > 0:
            affinities = affinities / np.sum(affinities)
            
        # Compute gradient pull
        pull = 0
        for target_cid, affinity in zip(valid_targets, affinities):
            target_emb = np.array(channel_embeddings_dict[target_cid])
            target_direction = target_emb - centroid
            norm_target = np.linalg.norm(target_direction)
            if norm_target > 0:
                cos_sim = np.dot(vid_movement, target_direction) / (norm_movement * norm_target)
                pull += affinity * cos_sim
                
        affinity_scores.append(pull)

    if affinity_scores:
        measurements_df['audience_adjacent_pull'] = affinity_scores
        print("Calculated audience adjacent pull.")


## Persist Artifacts

In [ ]:
import os
artifact_root = Path('/content/drive/MyDrive/Graphiko/analysis/channel_video_manifold/latest')
# Use exist_ok and handle environments where Drive is not mounted
try:
    artifact_root.mkdir(parents=True, exist_ok=True)
    measurements_df.to_csv(artifact_root / 'video_manifold_measurements.csv', index=False)
    print(f"Artifacts saved to {artifact_root}")
except Exception as e:
    print(f"Could not save to Drive: {e}")
    # Save locally as fallback
    os.makedirs('channel_video_manifold', exist_ok=True)
    measurements_df.to_csv('channel_video_manifold/video_manifold_measurements.csv', index=False)
    print("Artifacts saved locally to channel_video_manifold/")
